# inference.ipynb

Notebook ini mendemonstrasikan inferensi lokal, explainability SHAP, dan narasi Bahasa Indonesia.

Dataset kerja saat ini bersifat sintetis; lihat `data/processed/data_provenance.json` dan `models/robustness.json` untuk caveat validitas.


In [ ]:
from pathlib import Path
import pandas as pd
from src.explain import explain_single, get_explainer, shap_counterfactual
from src.narrative import render_explanation_narrative

assert Path("models/xgb_model.ubj").exists(), "Jalankan training.ipynb terlebih dahulu atau siapkan model lokal"
X_test = pd.read_parquet("test_data/features.parquet")
row = X_test.iloc[[0]]
row.head()


In [ ]:
import xgboost as xgb
model = xgb.Booster()
model.load_model("models/xgb_model.ubj")
explainer = get_explainer(model)
explanation = explain_single(row, model=model, explainer=explainer)
explanation


In [ ]:
counterfactual = shap_counterfactual(explanation, target_class=0)
render_explanation_narrative(explanation, counterfactual)


In [ ]:
import numpy as np
import onnxruntime as rt

if Path("models/xgb_model.onnx").exists():
    sess = rt.InferenceSession("models/xgb_model.onnx")
    input_name = sess.get_inputs()[0].name
    sample = row.to_numpy(dtype=np.float32)
    outputs = sess.run(None, {input_name: sample})
    outputs
else:
    print("models/xgb_model.onnx belum tersedia")
